# Unión de Conjuntos de Datos: Operación *Join*

Para realizar cualquier tipo de manipulación o análisis de datos complejo e interesante, a menudo es necesario reunir información procedente de varios conjuntos de datos a través de una operación de **Join** (Unión).

Esta es una técnica fundamental y muy conocida en el lenguaje SQL. Al realizar un *Join*, se combinan las columnas de dos conjuntos de datos (que pueden ser tablas totalmente distintas o incluso la misma tabla reflejada). El DataFrame combinado resultante contendrá las columnas de ambos lados, lo que te permitirá realizar análisis cruzados profundos que serían imposibles de lograr analizando cada conjunto de datos de forma individual.

---

## Ejemplo Práctico

Imaginemos que estamos trabajando con los dos siguientes conjuntos de datos en un entorno comercial:

### 1. Tabla de Clientes (`df_clientes`)
Contiene los datos demográficos y de perfil de cada cliente registrado en el sistema.

| id | edad | categoria_tarjeta | tipo_tarjeta |
| :---: | :---: | :--- | :---: |
| 01 | 40 | Bronce | D |
| 02 | 50 | Plata | C |
| 03 | 34 | Oro | B |
| 04 | 50 | Diamante | A |
| 05 | 26 | Bronce | D |
| 06 | 29 | Oro | B |
| 07 | 52 | Plata | C |
| 08 | 45 | Bronce | D |

### 2. Tabla de Compras (`df_compras`)
Contiene el historial de las transacciones o compras realizadas por los usuarios en la plataforma. *(Nota que el cliente `04` compró dos veces, el cliente `09` no está registrado en nuestro padrón, y algunos clientes no han comprado nada aún).*

| id | monto | fecha |
| :---: | :---: | :---: |
| 04 | 100 | 15-06-2026 |
| 04 | 98 | 16-06-2026 |
| 02 | 66 | 17-06-2026 |
| 05 | 150 | 17-06-2026 |
| 08 | 200 | 18-06-2026 |
| 06 | 104 | 18-06-2026 |
| 09 | 245 | 19-06-2026 |
| 01 | 310 | 19-06-2026 |

---

## El Poder del *Join*



Si realizamos un **Join** en estas dos tablas utilizando el campo común **`id`** como clave de unión, Spark alineará los registros correspondientes y generará un DataFrame unificado.

### ¿Qué respuestas de negocio podemos extraer tras el Join?
Al cruzar ambas fuentes de información, podemos responder preguntas analíticas de alto valor como:

* **Análisis demográfico de consumo:** ¿Cuál es la edad promedio de los clientes cuyas compras poseen los montos más elevados?
* **Optimización de productos financieros:** ¿Qué tipo de tarjeta (`A`, `B`, `C` o `D`) suele utilizarse para pagar los montos más pequeños o transacciones del día a día?
* **Comportamiento por segmentos:** ¿Los clientes con categoría `Diamante` gastan más dinero por transacción en comparación con los de categoría `Bronce`?



# Componentes y Tipos de *Join* en Spark SQL

Para realizar cualquier tipo de unión (*Join*) entre dos conjuntos de datos, Spark requiere obligatoriamente que especifiquemos **dos piezas de información**:

1. **La expresión de unión (*Join Expression*):** Especifica qué columnas de cada conjunto de datos deben compararse para determinar qué filas de ambos lados están relacionadas y se deben emparejar.
2. **El tipo de unión (*Join Type*):** Determina qué registros se deben incluir en el DataFrame final en caso de que haya o no coincidencias entre los datos.

---

## Tipos de *Join* admitidos en Spark SQL

A continuación se detallan los diferentes tipos de uniones que podemos especificar en el parámetro `how` de la función `.join()` de Spark:



### 1. `inner` (Inner Join)
Es el tipo de unión por defecto. Devuelve las filas de ambos conjuntos de datos **únicamente cuando la expresión de *join* se evalúa como verdadera** (es decir, cuando hay una coincidencia exacta en ambos lados).

### 2. `left` o `left_outer` (Left Outer Join)
Devuelve todas las filas del conjunto de datos de la **izquierda**, incluso si la expresión de *join* se evalúa como falsa (no hay coincidencia). Para las filas sin coincidencia en el lado derecho, Spark rellenará las columnas correspondientes con valores nulos (`Null`).

### 3. `right` o `right_outer` (Right Outer Join)
Funciona a la inversa del anterior. Devuelve todas las filas del conjunto de datos de la **derecha**, incluso cuando no hay coincidencias en el lado izquierdo. Las columnas del lado izquierdo que queden vacías se rellenarán con valores nulos (`Null`).

### 4. `outer` o `full_outer` (Full Outer Join)
Devuelve **todas las filas de ambos conjuntos de datos**, independientemente de si coinciden o no. Si un registro solo existe en el lado izquierdo, las columnas del derecho se muestran como `Null`, y viceversa.

### 5. `left_anti` (Left Anti Join)
Devuelve **únicamente las filas del conjunto de datos de la izquierda que NO tienen ninguna coincidencia** en el conjunto de datos de la derecha. El DataFrame resultante solo conservará las columnas del lado izquierdo.

### 6. `left_semi` (Left Semi Join)
Es el opuesto al Anti Join. Devuelve las filas del conjunto de datos de la izquierda **que SÍ tienen una coincidencia** en el de la derecha. A diferencia de un `inner join`, el `left_semi` **no une las columnas del lado derecho**; funciona de manera similar a un filtro de tipo "existe en".

### 7. `cross` (Cross Join / Producto Cartesiano)
Devuelve la combinación de **cada fila del conjunto de la izquierda con todas y cada una de las filas del de la derecha**. El número total de filas resultantes será exactamente el producto del tamaño de ambos conjuntos de datos ($N \times M$).

> ⚠️ **Advertencia de rendimiento:** El `cross` join debe evitarse o utilizarse con extrema precaución en Big Data, ya que puede generar miles de millones de filas innecesarias y agotar los recursos de memoria del clúster de forma instantánea.


In [1]:
!pip install findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Inner Join").getOrCreate()

In [6]:
empleados = spark.read.parquet("./data/empleados")
empleados.show()

+------+--------+
|nombre|num_dpto|
+------+--------+
|  Luis|      33|
| Katia|      33|
|  Raul|      34|
| Pedro|       0|
| Laura|      34|
|Sandro|      31|
+------+--------+



In [5]:
departamentos = spark.read.parquet("./data/departamentos")
departamentos.show()

+---+-----------+
| id|nombre_dpto|
+---+-----------+
| 31|     letras|
| 33|    derecho|
| 34| matemática|
| 35|informática|
+---+-----------+



## InnerJoin()

Este es el tipo de Join mas utilizado contendra las filas solo cuando la expresion de join se evalue como verdadera (cuando los valores de la columna del Join sean los mismos en ambos conjuntos de datos)

las filas que no tengan valores de columnas concidentes se excluiran del conjunto de datos resultante en spark SQL el inner Join es el tipo de join predeterminado por lo que es opcional especificarlo o no

In [8]:
from pyspark.sql.functions import col

In [9]:
join_df = empleados.join(departamentos, col("num_dpto") == col("id"))

join_df.show()

+------+--------+---+-----------+
|nombre|num_dpto| id|nombre_dpto|
+------+--------+---+-----------+
|  Luis|      33| 33|    derecho|
| Katia|      33| 33|    derecho|
|  Raul|      34| 34| matemática|
| Laura|      34| 34| matemática|
|Sandro|      31| 31|     letras|
+------+--------+---+-----------+



In [10]:
# si observamos lo que realiza el inner join es que obtiene los valroes en el df de la izq y el df de la derecha

Es lo que llamamos una intersection si nos fijamos el numero de dpto 0 de Pedro lo elimino porque no coincide en ninguno de los departamentos

como podemos ver no hubo necesidad de identificar el tipo de join ya que el InnerJoin es el join por defecto en spark

In [12]:
# especificando el tipo de join se hace al final de la consulta

join_df = empleados.join(departamentos, col("num_dpto") == col("id"), "inner")

join_df.show()

+------+--------+---+-----------+
|nombre|num_dpto| id|nombre_dpto|
+------+--------+---+-----------+
|  Luis|      33| 33|    derecho|
| Katia|      33| 33|    derecho|
|  Raul|      34| 34| matemática|
| Laura|      34| 34| matemática|
|Sandro|      31| 31|     letras|
+------+--------+---+-----------+



In [13]:
## Alternativa

join_df = empleados.join(departamentos).where(col("num_dpto") == col("id"))

join_df.show()

+------+--------+---+-----------+
|nombre|num_dpto| id|nombre_dpto|
+------+--------+---+-----------+
|  Luis|      33| 33|    derecho|
| Katia|      33| 33|    derecho|
|  Raul|      34| 34| matemática|
| Laura|      34| 34| matemática|
|Sandro|      31| 31|     letras|
+------+--------+---+-----------+

